# Exploration des agendas OpenAgenda — Pays de la Loire et régions comparables

**Objectif de ce notebook**

Le corpus initial du projet (4 sources agrégées) contenait **93 événements culturels** pour la région Pays de la Loire.
Ce notebook a pour but de juger si ce volume est suffisant pour l'étude et la présentation, en explorant deux pistes :

1. **Élargir la couverture Pays de la Loire** : existe-t-il d'autres agendas OpenAgenda (par ville, par thématique) non encore exploités qui pourraient enrichir le corpus ?
2. **Comparer avec d'autres régions françaises** : le volume de 93 événements est-il dans la norme, ou d'autres régions offrent-elles des agendas nettement plus fournis sur OpenAgenda ?

Ce notebook est un outil d'**exploration manuelle** : il n'écrit rien dans `data/`, ne modifie pas le pipeline existant (`fetch_events.py`, `preprocess_events.py`), et n'affecte donc pas le corpus déjà validé par tes tests. Une fois les résultats analysés, nous déciderons s'il vaut la peine d'ajouter une ou plusieurs sources dans `fetch_events.py`.

**Prérequis** : ce notebook utilise l'API OpenAgenda comme le reste du projet. Il faut donc un fichier `.env` à la racine du projet contenant `OPENAGENDA_API_KEY=...` (le même que celui déjà utilisé par `fetch_events.py`).


In [1]:
import os
import time
import requests
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("OPENAGENDA_API_KEY")
assert API_KEY, "OPENAGENDA_API_KEY manquant : vérifie ton fichier .env"

BASE_URL = "https://api.openagenda.com/v2"


## Fonctions utilitaires

Les mêmes briques que dans `explorer_agendas.py`, regroupées ici pour l'exploration :

- `search_agendas(query, size)` : recherche des agendas par mot-clé.
- `get_agenda_detail(uid)` : détail d'un agenda (nom, description, nombre total d'événements déclaré).
- `count_events(agenda_uid, ..., relative="upcoming")` : compte les événements à venir d'un agenda, avec filtres optionnels.


In [3]:
def search_agendas(query, size=10):
    """Recherche des agendas OpenAgenda par mot-clé, retourne une liste de dicts (uid, title, eventCount)."""
    resp = requests.get(
        f"{BASE_URL}/agendas",
        params={"key": API_KEY, "search": query, "size": size},
    )
    resp.raise_for_status()
    data = resp.json()
    results = []
    for agenda in data.get("agendas", []):
        results.append({
            "uid": agenda.get("uid"),
            "title": agenda.get("title", {}).get("fr") if isinstance(agenda.get("title"), dict) else agenda.get("title"),
            "eventCount": agenda.get("eventCount"),
        })
    return results


def count_events(agenda_uid, admin_level1=None, admin_level2=None, relative="upcoming"):
    """Compte les événements d'un agenda avec filtres optionnels (région, département), sans tout télécharger."""
    params = {"key": API_KEY, "size": 1, "detailed": 0}
    if relative:
        params["relative[]"] = relative
    if admin_level1:
        params["adminLevel1[]"] = admin_level1
    if admin_level2:
        params["adminLevel2[]"] = admin_level2

    resp = requests.get(f"{BASE_URL}/agendas/{agenda_uid}/events", params=params)
    resp.raise_for_status()
    data = resp.json()
    return data.get("total", 0)

#Cette méthode utilise search_agendas
def explore_search_terms(terms, relative="upcoming", pause=0.3):
    """Pour une liste de mots-clés, cherche les agendas correspondants et compte leurs événements à venir.
    Retourne un DataFrame trié par nombre d'événements décroissant."""
    rows = []
    seen_uids = set()
    for term in terms:
        agendas = search_agendas(term, size=5)
        for a in agendas:
            uid = a["uid"]
            if uid in seen_uids:
                continue
            seen_uids.add(uid)
            try:
                n = count_events(uid, relative=relative)
            except Exception as e:
                n = None
            rows.append({"terme_recherche": term, "uid": uid, "titre_agenda": a["title"], "evenements_a_venir": n})
        time.sleep(pause)
    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values("evenements_a_venir", ascending=False, na_position="last").reset_index(drop=True)
    return df


## 1. Rappel : volumes des 4 sources actuellement utilisées

Pour référence, voici le nombre d'événements à venir dans les 4 agendas déjà agrégés dans `fetch_events.py`, avant tout filtrage/nettoyage (le corpus final après nettoyage est de 93 événements).


In [13]:
sources_actuelles = {
    "Agenda de la Région des Pays de la Loire": "16676449",
    "Théâtre de Laval - CDN": "31651509",
    "Réseau des médiathèques & Archives du Mans": "48454528",
    "Unidivers (Grand Ouest)": "14115607",
}

rows = []
for nom, uid in sources_actuelles.items():
    try:
        n = count_events(uid, relative="upcoming")
    except Exception as e:
        n = f"Erreur: {e}"
    rows.append({"source": nom, "uid": uid, "evenements_a_venir": n})

df_actuel = pd.DataFrame(rows)
df_actuel


,source,uid,evenements_a_venir
0,Agenda de la Région des Pays de la Loire,16676449,115
1,Théâtre de Laval - CDN,31651509,55
2,Réseau des médiathèques & Archives du Mans,48454528,52
3,Unidivers (Grand Ouest),14115607,253


## 2. Recherche d'agendas Pays de la Loire non encore exploités

On élargit la recherche à des mots-clés couvrant les grandes villes et thématiques culturelles de la région, pour voir si d'autres agendas OpenAgenda existent et quel volume d'événements à venir ils proposent.

Ces agendas peuvent se recouper avec ceux déjà utilisés (mêmes événements, uid différents) — ce notebook ne fait pas de déduplication inter-agendas, il sert uniquement à repérer des pistes à explorer plus en détail ensuite.


### a. Recherche d'agendas par mots-clés

In [5]:
termes_pdl = [
    "Nantes culture",
    "Angers culture",
    "Le Mans culture",
    "La Roche-sur-Yon culture",
    "Laval culture",
    "Cholet culture",
    "Saumur",
    "Sablé-sur-Sarthe",
    "théâtre Pays de la Loire",
    "musique Pays de la Loire",
    "spectacle vivant Pays de la Loire",
    "musée Pays de la Loire",
    "médiathèque Pays de la Loire",
    "festival Pays de la Loire",
    "Nantes Métropole",
    "Angers Loire Métropole",
    "Le Mans Métropole",
]

df_pdl = explore_search_terms(termes_pdl, relative="upcoming")
df_pdl


,terme_recherche,uid,titre_agenda,evenements_a_venir
0,Nantes Métropole,82470621,Nantes Métropole,1640
1,Le Mans Métropole,50522407,Toulouse Métropole,1479
2,Le Mans culture,60871835,Culture Versailles,440
3,La Roche-sur-Yon culture,84178045,La culture en continu,273
4,Nantes culture,2363867,Île de Nantes,244
5,Cholet culture,94371477,Culture Châtellerault,214
6,Saumur,16676449,Agenda de la Région des Pays de la Loire,115
7,Angers culture,6008621,Culture Nevers,104
8,Nantes culture,66727730,Conservatoire de Nantes,103
9,Nantes Métropole,26188004,Vie associative Nantes ville et métropole,98


In [6]:
# Aperçu des agendas les plus prometteurs (hors sources déjà utilisées)
uids_deja_utilises = set(sources_actuelles.values())
df_pdl_nouveaux = df_pdl[~df_pdl["uid"].isin(uids_deja_utilises)]
df_pdl_nouveaux.head(20)


,terme_recherche,uid,titre_agenda,evenements_a_venir
0,Nantes Métropole,82470621,Nantes Métropole,1640
1,Le Mans Métropole,50522407,Toulouse Métropole,1479
2,Le Mans culture,60871835,Culture Versailles,440
3,La Roche-sur-Yon culture,84178045,La culture en continu,273
4,Nantes culture,2363867,Île de Nantes,244
5,Cholet culture,94371477,Culture Châtellerault,214
6,Saumur,16676449,Agenda de la Région des Pays de la Loire,115
7,Angers culture,6008621,Culture Nevers,104
8,Nantes culture,66727730,Conservatoire de Nantes,103
9,Nantes Métropole,26188004,Vie associative Nantes ville et métropole,98


Clairement, on constate que le seul candidat valable pour notre extension est Nantes Metropole car cet agenda concerne veritablement la ville de Nantes et donc les Pays de la loire. Cela n'est pas le cas des autres résultats qui sont pour la plupart des faux positifs ('Toulouse Metropole' n'a rien à voir avec 'Le Mans Metro^pole')

### b. Vérification de l'agenda 'Nantes Métropole' avant inclusion

On va vérifier si les évenements de 'Nantes Metropole' concerne bien des évenements culturels et non pas d'autres types d'évenement.

In [10]:
AGENDA_UID = "82470621"  # Nantes Métropole

resp = requests.get(
    f"https://api.openagenda.com/v2/agendas/{AGENDA_UID}/events",
    params={
        "key": API_KEY,
        "size": 30,
        "detailed": 0,
        "relative[]": "upcoming",
        "monolingual": "fr",
    },
)
resp.raise_for_status()
data = resp.json()

print(f"Total événements à venir : {data.get('total')}\n")
for e in data.get("events", []):
    title = e.get("title", {})
    title = title.get("fr") if isinstance(title, dict) else title
    keywords = e.get("keywords", {})
    keywords = keywords.get("fr") if isinstance(keywords, dict) else keywords
    print(f"- {title}  {f'[{keywords}]' if keywords else ''}")

Total événements à venir : 1640

- Inauguration de la ressourcerie métropolitaine de Rezé  
- La Nuit du Théâtre 2026  
- Festival Midi Minuit Poésie  [['nantes', 'nantes métropole', 'midiminuitpoésie', 'maison de la poésie']]
- Les Utopiales 2026 – La billetterie est ouverte  [['nantes', 'nantes métropole', 'utopiales']]
- [Inscriptions imminentes] Marathon de Nantes + Semi-marathon + Foulées de l'éléphant 2027  
- Les lundis imprimés  [["techniques d'impression", 'tetra pak', 'sérigraphie', 'monotype', 'micro-édition', 'illustration', 'atelier créatif', 'diy', 'impression', 'nantes', 'rond-point de paris', 'adulte', 'ado', 'tous niveaux']]
- Journée Gratuite de sensibilisations – Nantes Digital Week  
- Salon 24 heures pour l’emploi et la formation  
- Sensibilisation : Que fait Google de vos photos ?  [['Google', 'google photos', 'photos', 'numérique']]
- Sensibilisation : Les objets connectés pour seniors  [['Google', 'google photos', 'photos', 'numérique']]
- Sensibilisation : Le 

On voit clairement dans cet aperçu que beaucoup d'evenements ne concernent pas des évenements culturels mais autre (sensibilisation, débat, conférence, etc, etc). On va donc analyser ce groupe plus finement pour savoir la proportion d'évenement culturels et non-culturels.

In [17]:
"""
Analyse de l'agenda "Nantes Métropole" (uid 82470621) : filtrage culturel
et répartition géographique des résultats, pour décider s'il vaut mieux
l'ajouter (filtré) plutôt que chercher un agenda dédié à la Vendée.
"""

import os
import re
import time
import unicodedata
from collections import Counter

import requests
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv("OPENAGENDA_API_KEY")
assert API_KEY, "OPENAGENDA_API_KEY manquant"

AGENDA_UID = "82470621"  # Nantes Métropole
BASE_URL = "https://api.openagenda.com/v2"

# Mots-clés considérés comme "culturels" (recherchés dans le titre + les mots-clés
# de l'événement). Liste construite à partir de l'échantillon déjà observé
# (théâtre, danse, musique, exposition...) + des thématiques classiques de la
# programmation culturelle. À ajuster si besoin après avoir vu les résultats.
MOTS_CLES_CULTURELS = [
    "theatre", "danse", "musique", "concert", "exposition", "expo",
    "festival", "spectacle", "cirque", "marionnette", "opera", "cinema",
    "conte", "poesie", "litterature", "chorale", "ballet", "mediatheque",
    "bibliotheque", "peinture", "sculpture", "photographie", "vernissage",
    "patrimoine", "art contemporain", "atelier creatif", "gravure",
    "illustration", "creation artistique", "arts plastiques", "slam",
    "lecture", "spectacle vivant", "humour", "conservatoire",
]

# Mots-clés qui, même s'ils recoupent un terme ci-dessus, signalent presque
# toujours un événement institutionnel/administratif plutôt que culturel
# (observé dans l'échantillon : ateliers numériques, emploi, sensibilisation...)
MOTS_CLES_EXCLUS = [
    "numerique", "rgpd", "senior", "recrutement", "emploi", "formation professionnelle",
    "entrepreneuriat", "sensibilisation", "repair cafe", "conseil municipal",
    "enquete publique", "consultation", "inscription", "marathon", "ressourcerie",
    "mobilite", "debat", "reunion publique", "permanence", "demarche en ligne",
]


def normalize(text):
    if not text:
        return ""
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("ascii")
    return text.lower()


def is_culturel(title, keywords):
    texte = normalize(title) + " " + normalize(" ".join(keywords or []))
    if any(mot in texte for mot in MOTS_CLES_EXCLUS):
        return False
    return any(mot in texte for mot in MOTS_CLES_CULTURELS)


def fetch_all_events(agenda_uid, size=100, pause=0.2):
    """Récupère tous les événements à venir d'un agenda, avec pagination."""
    events = []
    after = None
    while True:
        params = {
            "key": API_KEY,
            "size": size,
            "detailed": 0,
            "relative[]": "upcoming",
            "monolingual": "fr",
        }
        if after:
            params["after[]"] = after

        resp = requests.get(f"{BASE_URL}/agendas/{agenda_uid}/events", params=params)
        resp.raise_for_status()
        data = resp.json()

        batch = data.get("events", [])
        events.extend(batch)

        after = data.get("after")
        if not after or not batch:
            break
        time.sleep(pause)

    return events


DEPARTEMENTS_PAR_CODE_POSTAL = {
    "44": "Loire-Atlantique",
    "49": "Maine-et-Loire",
    "53": "Mayenne",
    "72": "Sarthe",
    "85": "Vendée",
}


def department_from_postal_code(postal_code):
    if not postal_code or len(postal_code) < 2:
        return None
    return DEPARTEMENTS_PAR_CODE_POSTAL.get(postal_code[:2], f"Autre ({postal_code[:2]})")


def extract_fields(event):
    title = event.get("title", {})
    title = title.get("fr") if isinstance(title, dict) else title

    keywords = event.get("keywords", {})
    keywords = keywords.get("fr") if isinstance(keywords, dict) else keywords

    location = event.get("location", {})
    postal_code = location.get("postalCode") if isinstance(location, dict) else None
    city = location.get("city") if isinstance(location, dict) else None
    department = department_from_postal_code(postal_code)

    return title, keywords or [], department, city


def main():
    print(f"Récupération de tous les événements à venir de l'agenda {AGENDA_UID}...")
    events = fetch_all_events(AGENDA_UID)
    print(f"{len(events)} événements récupérés au total\n")

    # DEBUG : structure du champ location du premier événement
    #if events:
    #    import json
    #    print("DEBUG - location du 1er événement :")
    #    print(json.dumps(events[0].get("location", {}), ensure_ascii=False, indent=2))
    #    print()
    
    dept_total = Counter()
    dept_culturel = Counter()
    n_culturel = 0

    for event in events:
        title, keywords, department, city = extract_fields(event)
        dept_key = department or "Inconnu"
        dept_total[dept_key] += 1

        if is_culturel(title, keywords):
            n_culturel += 1
            dept_culturel[dept_key] += 1

    print(f"Événements classés culturels : {n_culturel} / {len(events)} "
          f"({100 * n_culturel / len(events):.1f}%)\n")

    print("Répartition par département (TOUS les événements) :")
    for dept, n in dept_total.most_common():
        print(f"  {dept:25s} {n}")

    print("\nRépartition par département (événements CULTURELS uniquement) :")
    for dept, n in dept_culturel.most_common():
        print(f"  {dept:25s} {n}")


if __name__ == "__main__":
    main()

Récupération de tous les événements à venir de l'agenda 82470621...
1645 événements récupérés au total

DEBUG - location du 1er événement :
{
  "address": "7 rue Suzanne Noël 44400",
  "city": "Rezé",
  "latitude": 47.154699,
  "name": "Ressourcerie métropolitaine",
  "longitude": -1.547221
}

Événements classés culturels : 728 / 1645 (44.3%)

Répartition par département (TOUS les événements) :
  Inconnu                   1645

Répartition par département (événements CULTURELS uniquement) :
  Inconnu                   728


On constate que 'Nantes Metropole' contient plus de la moitié d'evenements non culturels, ce qui n'en fait pas un bon candidat en raison du filtrage qui va devoir être du fait et du peu de garantie sur le résultat de ce filtrage. En effet, en l'absence de catégorie claire 'Culturel'/'Non culturel', il est difficile de statuer sur le caractère culturel d'un évenement sans prendre des risques. Par conséquent, mieux trouver des agendas plus petits, mais pour lesquels on est sur de leur contenu (ou alors on maitrise mieux leur contenu compte-tenu du peu d'évenement). 

### c. Vérification de l'agenda 'Le Théâtre de Laval – CDN' et 'Réseau des médiathèques & Archives du Mans'

Voici justement deux agendas qui concernent un perimètre limité (Laval et Le Mans) et un nombre restreint d'évenements. Par ailleurs, les deux départements auxquels ils font référence sont ceux pour lesquels nous n'avions pas d'évenements, ce qui en fait de très bons candidats.

In [12]:
AGENDAS = {
    "Le Théâtre de Laval - CDN (Mayenne)": "31651509",
    "Réseau des médiathèques & Archives du Mans (Sarthe)": "48454528",
}
 
 
def sample_agenda(nom, uid, size=30):
    resp = requests.get(
        f"https://api.openagenda.com/v2/agendas/{uid}/events",
        params={
            "key": API_KEY,
            "size": size,
            "detailed": 0,
            "relative[]": "upcoming",
            "monolingual": "fr",
        },
    )
    resp.raise_for_status()
    data = resp.json()
 
    print("=" * 70)
    print(f"{nom} (uid {uid})")
    print(f"Total événements à venir : {data.get('total')}")
    print("-" * 70)
    for e in data.get("events", []):
        title = e.get("title", {})
        title = title.get("fr") if isinstance(title, dict) else title
        location = e.get("location", {})
        city = location.get("city") if isinstance(location, dict) else None
        keywords = e.get("keywords", {})
        keywords = keywords.get("fr") if isinstance(keywords, dict) else keywords
        print(f"- {title}  ({city})  {f'[{keywords}]' if keywords else ''}")
    print()
 
 
for nom, uid in AGENDAS.items():
    sample_agenda(nom, uid)
 

Le Théâtre de Laval - CDN (Mayenne) (uid 31651509)
Total événements à venir : 55
----------------------------------------------------------------------
- [EXPOSITION] Un voyage en Rumba · Kkrist Mirror  (Laval)  [['exposition']]
- Éclairages et clés de lecture : Qui es-tu William Shakespeare ?  (Laval)  [['littérature', 'le roi lear', 'théâtre']]
- Le Roi Lear  (Laval)  [['#Avec restauration', 'Audiodescription', 'Théâtre']]
- La Mòssa + Pérégrines  (Laval)  [['#Avec restauration', 'Musique']]
- [EXPOSITION] Capharnaüm · François Soutif  (Laval)  [['exposition']]
- Murmures de la forêt  (Laval)  [['-> Pass famille', '#Sans restauration', '']]
- Ballet Bar  (Laval)  [['-> Pass famille', '#Avec restauration', 'Danse']]
- Ateliers !  (Laval)  
- La Renverse  (Laval)  [['-> Pass famille', '#Sans restauration', 'Marionnette et formes manipulées']]
- [PUPAZZI] La Veillée des Lucioles · Cie Les Grandes Personnes  (Laval)  [['déambulation', 'festival pupazzi']]
- Boîte crânienne  (Laval)  [['#

Bingo ! Les évenements sont tous culturels et ils concernent deux départements pour lesquels nous n'avons pas d'évenements. Nous les conservons.

### d. Exploration des agendas qui concernent la vendée (culturels et nombre d'évenements)

Derniere zone d'ombre : nous n'avons qu'un évenement qui concerne le département de la vendée dans le Pays de la loire. Par conséquent, nous essayons de trouver un candidat qui concerne la vendée grâce à des mots clés de villes vendéennes (au même titre que le théâtre de Laval pour Laval et Réseau des médiathèques & Archives du Mans pour Le Mans).

In [18]:
TERMES_VENDEE = [
    "La Roche-sur-Yon culture",
    "Les Sables-d'Olonne culture",
    "Challans culture",
    "Fontenay-le-Comte culture",
    "Vendée culture",
    "théâtre Vendée",
    "médiathèque Vendée",
    "musée Vendée",
    "festival Vendée",
    "spectacle vivant Vendée",
    "Historial de la Vendée",
    "Vendée Culture",
    "La Roche-sur-Yon Agglomération",
    "Pays Yon et Vie",
]
 
 
def search_agendas(query, size=8):
    resp = requests.get(
        f"{BASE_URL}/agendas",
        params={"key": API_KEY, "search": query, "size": size},
    )
    resp.raise_for_status()
    data = resp.json()
    results = []
    for agenda in data.get("agendas", []):
        title = agenda.get("title", {})
        title = title.get("fr") if isinstance(title, dict) else title
        results.append({"uid": agenda.get("uid"), "titre": title})
    return results
 
 
def count_events(agenda_uid, relative="upcoming"):
    resp = requests.get(
        f"{BASE_URL}/agendas/{agenda_uid}/events",
        params={"key": API_KEY, "size": 1, "detailed": 0, "relative[]": relative},
    )
    resp.raise_for_status()
    return resp.json().get("total", 0)
 
 
def sample_titles(agenda_uid, size=5):
    resp = requests.get(
        f"{BASE_URL}/agendas/{agenda_uid}/events",
        params={"key": API_KEY, "size": size, "detailed": 0, "relative[]": "upcoming", "monolingual": "fr"},
    )
    resp.raise_for_status()
    titres = []
    for e in resp.json().get("events", []):
        t = e.get("title", {})
        titres.append(t.get("fr") if isinstance(t, dict) else t)
    return titres
 
 
def main():
    rows = []
    seen = set()
    for terme in TERMES_VENDEE:
        for a in search_agendas(terme):
            uid = a["uid"]
            if uid in seen:
                continue
            seen.add(uid)
            try:
                n = count_events(uid)
            except Exception:
                n = None
            rows.append({"terme_recherche": terme, "uid": uid, "titre_agenda": a["titre"], "evenements_a_venir": n})
        time.sleep(0.3)
 
    df = pd.DataFrame(rows).sort_values("evenements_a_venir", ascending=False, na_position="last").reset_index(drop=True)
    print(df.to_string())
 
    print("\n--- Aperçu des 5 agendas les plus prometteurs (aperçu de titres) ---\n")
    for _, row in df.head(5).iterrows():
        print(f"### {row['titre_agenda']} (uid {row['uid']}, {row['evenements_a_venir']} évts à venir)")
        try:
            for t in sample_titles(row["uid"]):
                print(f"  - {t}")
        except Exception as e:
            print(f"  (erreur : {e})")
        print()
 
 
if __name__ == "__main__":
    main()

                   terme_recherche       uid                                                                titre_agenda  evenements_a_venir
0         La Roche-sur-Yon culture  60871835                                                          Culture Versailles                 447
1   La Roche-sur-Yon Agglomération  78891697                                                          Alès Agglomération                 377
2        Fontenay-le-Comte culture   7562544                                                              Agenda de test                 338
3               médiathèque Vendée  81756682                                                   Villeurbanne Médiathèques                 291
4         La Roche-sur-Yon culture  84178045                                                       La culture en continu                 273
5   La Roche-sur-Yon Agglomération  93399464                                   Communauté d'agglomération de l'Albigeois                 270
6         La 

Encore une fois : il n'y a presque que des faux positifs qui concerne des villes et des localités qui n'ont rien à voir avec la Vendée... Sauf 'Département de la Vendée' que nous allons explorer pour vérification avec insertion.

### e. Exploration de l'agenda 'Département de la Vendée'

In [19]:
load_dotenv()
API_KEY = os.getenv("OPENAGENDA_API_KEY")
assert API_KEY, "OPENAGENDA_API_KEY manquant"

AGENDA_UID = "7894666"  # Département de la Vendée

resp = requests.get(
    f"https://api.openagenda.com/v2/agendas/{AGENDA_UID}/events",
    params={
        "key": API_KEY,
        "size": 63,
        "detailed": 0,
        "relative[]": "upcoming",
        "monolingual": "fr",
    },
)
resp.raise_for_status()
data = resp.json()

print(f"Total événements à venir : {data.get('total')}\n")

villes = Counter()
for e in data.get("events", []):
    title = e.get("title", {})
    title = title.get("fr") if isinstance(title, dict) else title
    keywords = e.get("keywords", {})
    keywords = keywords.get("fr") if isinstance(keywords, dict) else keywords
    location = e.get("location", {})
    city = location.get("city") if isinstance(location, dict) else None
    villes[city or "Inconnu"] += 1
    print(f"- {title}  ({city})  {f'[{keywords}]' if keywords else ''}")

print("\nRépartition par ville :")
for ville, n in villes.most_common():
    print(f"  {ville:25s} {n}")

Total événements à venir : 63

- Réalise ton caligramme  (Saint-André-Treize-Voies)  [['Bibliothèques de Montréverd']]
- Automobiliste, cycliste ou piéton, quoi de neuf sur la route ?  (La Bernardière)  
- Vendredi 25 septembre 2026  à 20h Salle du Cercle  (Montaigu-Vendée)  [['Echanges et Solidarité']]
- Les Wriggles  (Montaigu)  [['Théâtre de Thalie', 'Terres de Montaigu']]
- Boissière Futsal - 1er match de la saison 2026/2027  (La Boissière-de-Montaigu)  [['Boissière Fustal']]
- Repas à emporter  (Montréverd)  [['UNC Saint-Sulpice-le-Verdon']]
- Trail de Nantes à Montaigu  (Montaigu)  
- Spectacle les Tisseuses d'étoiles  (Rocheservière)  [['Site Saint Sauveur']]
- Les tisseuses d'étoiles  (Rocheservière)  [['médiathèque de Rocheservière']]
- Conseil municipal de Montaigu-Vendée - Septembre 2026  (Montaigu)  
- Dictée pour tous - Octobre 2026 - L'Herbergement  (L'Herbergement)  [["Club Sourire d'Automne"]]
- Le Voyage de monsieur Perrichon  (Montaigu)  [['Théâtre de Thalie', 'Terres

'Département de la vendée' semble être un bon candidat, cependant il va falloir le filtrer car il contient une petite proportion d'évenement non culturels (Ex : Vide Grenier  (Treize-Septiers). C'est ce que nous ferons dans le script de pre-processing (preprocess_events.py) avant la vectorisation.

## 3. Comparaison avec d'autres régions françaises

Pour situer le volume de la région Pays de la Loire, on recherche l'agenda régional officiel (motif `"Agenda de la Région {région}"`) d'une dizaine d'autres régions françaises et on compare leur nombre d'événements à venir.

Cette comparaison est indicative : elle suppose que chaque région dispose d'un agenda structuré comme celui des Pays de la Loire, ce qui n'est pas garanti (certaines régions n'ont peut-être pas d'agenda officiel actif sur OpenAgenda, ou utilisent un nom différent).


In [7]:
regions = [
    "Pays de la Loire",
    "Bretagne",
    "Nouvelle-Aquitaine",
    "Occitanie",
    "Auvergne-Rhône-Alpes",
    "Provence-Alpes-Côte d'Azur",
    "Île-de-France",
    "Grand Est",
    "Hauts-de-France",
    "Normandie",
    "Bourgogne-Franche-Comté",
    "Centre-Val de Loire",
]

rows = []
for region in regions:
    query = f"Agenda de la Région {region}"
    agendas = search_agendas(query, size=3)
    if not agendas:
        rows.append({"region": region, "uid": None, "titre_agenda": None, "evenements_a_venir": None})
        continue
    # on prend le premier résultat (le plus pertinent)
    a = agendas[0]
    try:
        n = count_events(a["uid"], relative="upcoming")
    except Exception:
        n = None
    rows.append({"region": region, "uid": a["uid"], "titre_agenda": a["title"], "evenements_a_venir": n})
    time.sleep(0.3)

df_regions = pd.DataFrame(rows).sort_values("evenements_a_venir", ascending=False, na_position="last").reset_index(drop=True)
df_regions


,region,uid,titre_agenda,evenements_a_venir
0,Hauts-de-France,8407019,Région Hauts-de-France,1951
1,Pays de la Loire,16676449,Agenda de la Région des Pays de la Loire,115
2,Grand Est,16676449,Agenda de la Région des Pays de la Loire,115
3,Bretagne,16676449,Agenda de la Région des Pays de la Loire,115
4,Normandie,16676449,Agenda de la Région des Pays de la Loire,115
5,Nouvelle-Aquitaine,97589740,Agenda L'Europe en Nouvelle-Aquitaine,20
6,Occitanie,83421114,Agenda de l'Europe en Occitanie,13
7,Provence-Alpes-Côte d'Azur,66808344,Agenda de l'Europe en Provence-Alpes-Côte-d'Azur,13
8,Auvergne-Rhône-Alpes,98276266,Agenda de l'Europe en Auvergne-Rhône-Alpes,8
9,Île-de-France,17344582,Chambre d'agriculture de région Ile-de-France,6


On voit en effet que la plupart des résultats sont des faux positifs et concerne la région Pays de la loire (de part la structure de la requête 'Agenda de la région {region}'). Pour les autres, ils concernent un nombre minime d'evenements. Par conséquent cette piste n'est pas intéressante.

## 4. Note : l'endpoint expérimental "lecture transverse"

OpenAgenda propose un endpoint expérimental `/v2/events` permettant d'interroger **tous les agendas de la plateforme en même temps** (au lieu d'agenda par agenda comme ci-dessus), avec les mêmes filtres géographiques (`adminLevel1[]`, etc.).

Cet endpoint n'est pas activé par défaut : il faut en faire la demande auprès du support OpenAgenda pour l'utiliser. S'il était activé, il permettrait de mesurer directement le nombre total d'événements culturels disponibles sur toute la plateforme pour la région Pays de la Loire, sans avoir à deviner quels agendas explorer un par un comme on vient de le faire.

Si le volume future (avec les modifications apportées suite à cette exploration) s'avère être une vraie limite pour l'étude, il faudrait contacter le support OpenAgenda pour activer cet endpoint est une piste à envisager pour une version future du projet — à mentionner éventuellement dans le rapport technique comme perspective d'amélioration.


## 5. Conclusion

L'exploration d'agendas d'autres régions ne semble pas pertinentes. En revanche, celle des agendas de localités en pays de la loire est plus fructueuse. Nous avons donc choisi de rajouter un agenda local par département peu représenté en terme de nombre d'évenements : 
- "Théâtre de Laval - CDN" pour la Mayenne.
- "Réseau des médiathèques & Archives du Mans" pour la Sarthe.
- "Département de la Vendée" pour la Véndée.

Les deux premiers agendas ne ramènent que des évenements culturels. Tandis que le dernier ramène des évenements non culturels. Par conséquent, celui-ci fera l'objet d'un filtrage spécifique pour supprimer les évenements non culturels. Pour cet agenda, nous appliquerons un traitement par liste blanche (et non par liste noir comme pour les deux premiers agendas) car le nombre d'évenements qu'il contient est faible. Par conséquent on peut se permettre d'analyser la totalité des évenements et ne garder que ceux qui concernent à coup sur des évenements culturels. Appliquer le filtrage par liste noire aurait laissé trop d'évenements non culturels dans cet agenda. Et inversement, appliquer ce filtrage (par liste blanche) aux deux premiers agendas aurait été beaucoup trop strict, de par leur grand nombre d'évenements, et aurait supprimé trop d'évenements. 

Les deux filtrages ne sont pas redondans mais complémentaires.
Nous en arrivons au final à 212 évenements avec la répartition suivante : 
| Département | Avant (4 sources) | Avec Laval+Mans | Avec Laval+Mans+Vendée filtrée |
|---|---|---|---|
| Loire-Atlantique | 57 | 51 | 51 |
| Maine-et-Loire | 34 | 32 | 32 |
| Mayenne | 0 | 55 | 55 |
| Sarthe | 0 | 52 | 52 |
| Vendée | 2 | 1 | 22 |
| **Total** | **93** | **191** | **212** |